
<a id='chap-coursdeep7'></a>

 Work in progress

Contenu rédigé à partir des diapositives coursSSL.pdf, faute de source
préexistante pour cette séance. Susceptible d’être ajusté.

Les informations pratiques concernant le déroulement de l’unité d’enseignement RCP209 « Apprentissage, réseaux de neurones et modèles graphiques » au Cnam se trouvent dans [ce préambule](preambule.html).

# XIV.1 - Cours

[[Diapositives du cours](http://cedric.cnam.fr/vertigo/Cours/ml2/docs/coursSSL.pdf)]

## Rappel : les paradigmes d’apprentissage

Jusqu’ici, nous avons presque exclusivement considéré l”**apprentissage supervisé** : étant donné un jeu de données étiqueté $ \mathcal D = \{(x_i, y_i)\}_{i=1}^N $, on cherche à apprendre une fonction de prédiction $ f_\theta : \mathcal X \rightarrow \mathcal Y $ en minimisant une fonction de coût supervisée $ \frac{1}{N}\sum_i \mathcal L(f_\theta(x_i), y_i) $.

Cette approche a permis des progrès spectaculaires, mais souffre de limitations importantes :

- les annotations sont **coûteuses** à produire (expertise humaine, temps), et leur coût croît linéairement avec la taille du jeu de données ;  
- les étiquettes peuvent être **bruitées, incohérentes ou ambiguës** ;  
- un modèle entraîné pour une tâche donnée apprend des représentations **spécifiques à cette tâche**, qui ne se transfèrent pas nécessairement bien à d’autres tâches ou d’autres jeux de données.  


Trois autres paradigmes tentent de s’affranchir, en tout ou partie, de la supervision :

- l”**apprentissage non-supervisé**, où l’on ne dispose que d’observations $ \mathcal D = \{x_i\}_{i=1}^N $ sans étiquette, et où l’on cherche à en apprendre la structure (partitionnement, réduction de dimension, estimation de densité) ;  
- l”**apprentissage semi-supervisé**, où l’on dispose d’un petit sous-ensemble étiqueté et d’un grand sous-ensemble non-étiqueté, combinés dans une même fonction de coût ;  
- l”**apprentissage auto-supervisé** (*self-supervised learning*, SSL), objet de cette séance.  

## Note

L’apprentissage non-supervisé « classique » (par exemple, un auto-encodeur qui minimise une erreur de reconstruction) souffre lui aussi d’une limitation : rien ne garantit que la représentation apprise soit utile pour une tâche donnée. Reconstruire parfaitement une image conserve toute son information, ce qui n’est ni nécessaire ni souhaitable pour, par exemple, la classifier : les détails redondants ou le bruit ne sont d’aucune utilité pour la sémantique de l’image.

## Apprentissage auto-supervisé et tâches prétextes

L’idée de l’apprentissage auto-supervisé est de **définir une tâche prétexte** (*pretext task*) dont les étiquettes $ y_i^{\text{pretext}} $ peuvent être générées automatiquement à partir des données elles-mêmes, sans aucune intervention humaine. On se ramène ainsi à un problème supervisé « classique » sur le jeu de données $ \mathcal D = \{(x_i, y_i^{\text{pretext}})\}_{i=1}^N $, mais l’objectif final n’est pas la tâche prétexte elle-même : on espère que la résoudre force le modèle à apprendre des représentations utiles, qui se **transfèrent** ensuite à la véritable tâche d’intérêt (la tâche *downstream*), par exemple par un simple apprentissage d’un classifieur linéaire sur les représentations gelées (*linear probing*), ou par un ré-entraînement complet du modèle sur peu de données (*fine-tuning*).

## Ce que fait ce TP

Le TP de cette séance met en œuvre un exemple d’apprentissage auto-supervisé contrastif (SimCLR), et évalue la qualité des représentations apprises en fonction de la quantité d’étiquettes disponibles pour la tâche *downstream*.

### Tâches prétextes historiques

Les premiers travaux en SSL pour la vision ont proposé des tâches prétextes fondées sur l’intuition, sans garantie théorique que les résoudre produise des représentations sémantiquement riches :

- **Prédiction de rotation** (Gidaris et al., 2018) : l’image est tournée d’un angle choisi parmi $ \{0°, 90°, 180°, 270°\} $, et le modèle doit prédire cet angle (classification à 4 classes). L’hypothèse est qu’un modèle ne peut reconnaître la bonne rotation que s’il a acquis une forme de « bon sens visuel » sur l’apparence normale des objets.  
- **Prédiction de position relative de patchs** (Doersch et al., 2015) : on extrait deux patchs d’une même image (un patch central et un patch voisin parmi 8 positions possibles), et le modèle doit prédire la position relative du second par rapport au premier.  
- **Puzzle (*jigsaw*)** (Noroozi et Favaro, 2016) : une image est découpée en 9 patchs puis mélangée selon une permutation tirée d’un ensemble fixé ; le modèle doit prédire quelle permutation a été appliquée.  
- **Inpainting (Context Encoders)** (Pathak et al., 2016) : une région rectangulaire de l’image est masquée, et un auto-encodeur doit la reconstruire à partir du contexte environnant, entraîné avec une combinaison d’une erreur de reconstruction et d’une perte adversariale.  
- **Colorisation** (Zhang et al., 2016) : à partir d’une image en niveaux de gris, le modèle doit prédire les canaux de chrominance de l’image couleur originale.  

### Limitations des tâches prétextes historiques

Ces approches partagent trois limitations importantes :

- elles encouragent souvent le modèle à exploiter des **raccourcis** (*shortcuts*) triviaux (artefacts de compression aux bords des patchs pour le puzzle, aberration chromatique pour la colorisation) plutôt qu’une compréhension sémantique du contenu ;  
- elles apprennent des représentations **spécifiques à la tâche prétexte choisie**, qui ne transfèrent que partiellement aux tâches d’intérêt (classification, détection, segmentation) ;  
- le choix de la tâche prétexte repose sur l’intuition du concepteur, sans garantie théorique que la résoudre produise des représentations généralement utiles.  


Ces limitations ont motivé le développement de familles de méthodes plus générales et aujourd’hui dominantes : l’apprentissage contrastif, l’auto-distillation, et la modélisation masquée.

## Apprentissage contrastif

### Principe

L’apprentissage contrastif définit un objectif unique et générique, indépendant du contenu des données : apprendre un espace de représentation dans lequel des exemples **similaires sont rapprochés**, et des exemples **dissimilaires sont éloignés**. Pour chaque exemple de référence (l”*ancre*) $ x $, on construit :

- un exemple **positif** $ x^+ $, similaire à $ x $ (typiquement, une autre vue de la même image obtenue par augmentation de données) ;  
- un ou plusieurs exemples **négatifs** $ x^- $, dissimilaires de $ x $ (typiquement, d’autres images du batch).  


On cherche alors une fonction $ f $ telle que le score de similarité entre représentations vérifie :

$$
\mathrm{score}(f(x), f(x^+)) \gg \mathrm{score}(f(x), f(x^-))~.
$$

La fonction de coût la plus utilisée est la perte **InfoNCE** (*Noise Contrastive Estimation*), qui formule ce problème comme une classification à $ N $ classes (retrouver l’exemple positif parmi $ N-1 $ exemples négatifs) :

$$
\mathcal L_{\text{InfoNCE}} = -\mathbb E_X\left[ \log \frac{\exp(s(f(x), f(x^+)))}{\exp(s(f(x), f(x^+))) + \sum_{j=1}^{N-1} \exp(s(f(x), f(x_j^-)))} \right]~,
$$

où $ s(\cdot, \cdot) $ est une fonction de similarité (par exemple, le cosinus). On reconnaît dans cette expression l’entropie croisée d’un classifieur softmax à $ N $ voies, dont le rôle est d’identifier l’exemple positif parmi les $ N $ candidats. On peut montrer que $ \mathcal L_{\text{InfoNCE}} $ est une borne inférieure de l’information mutuelle entre $ f(x) $ et $ f(x^+) $ : la minimiser revient donc à maximiser (une borne sur) cette information mutuelle.

### Question

Pourquoi le nombre d’exemples négatifs disponibles à chaque itération est-il un facteur important pour la qualité de l’apprentissage contrastif ?

### SimCLR

**SimCLR** (Chen et al., 2020) est une instanciation simple et efficace de ce principe :

1. Pour chaque image $ x $ d’un batch de taille $ N $, on génère deux vues augmentées $ \tilde x_i $ et $ \tilde x_j $ par composition aléatoire de transformations (recadrage aléatoire, distorsion des couleurs, mise en niveaux de gris, flou gaussien…). Ces deux vues forment une paire positive ; les $ 2(N-1) $ autres vues du batch servent d’exemples négatifs.  
1. Un réseau encodeur $ f(\cdot) $ (par exemple, un ResNet) calcule une représentation $ h = f(\tilde x) $ pour chaque vue.  
1. Une **tête de projection** $ g(\cdot) $, un petit MLP, projette cette représentation dans un espace de dimension plus faible où la perte contrastive est appliquée : $ z = g(h) $.  
1. On optimise la perte InfoNCE (ici appelée *NT-Xent*) sur les similarités cosinus entre les $ z $, pour toutes les paires du batch.  
1. Seul l’encodeur $ f(\cdot) $ est conservé pour les tâches *downstream* ; la tête de projection $ g(\cdot) $ est jetée après l’apprentissage.  

### Question

À votre avis, pourquoi conserve-t-on l’encodeur $ f(\cdot) $ pour les tâches *downstream*, plutôt que la représentation projetée $ z = g(f(x)) $ sur laquelle la perte contrastive est directement appliquée ?

Un constat empirique important de SimCLR est que la performance s’améliore avec la **taille du batch** (donc du nombre d’exemples négatifs disponibles) et la **durée de l’entraînement**, au prix d’un besoin en mémoire et en calcul plus important.

### Momentum Contrast (MoCo)

**MoCo** (He et al., 2020) répond à cette contrainte mémoire en découplant la taille du batch du nombre d’exemples négatifs disponibles. Plutôt que d’utiliser uniquement les autres exemples du batch courant comme négatifs, MoCo maintient une **file d’attente** (*queue*) de représentations calculées lors des itérations précédentes, ce qui permet un grand nombre de négatifs sans avoir à agrandir le batch. Ces représentations sont calculées par un encodeur « momentum », une copie de l’encodeur principal dont les poids $ \theta_k $ sont mis à jour non pas par descente de gradient mais par moyenne mobile exponentielle des poids de l’encodeur principal $ \theta_q $ :

$$
\theta_k \leftarrow m\,\theta_k + (1-m)\,\theta_q~,
$$

avec $ m $ proche de 1 (par exemple, 0,999). Cette mise à jour lente assure que les représentations de la file d’attente restent cohérentes entre elles malgré l’absence de rétropropagation du gradient à travers l’encodeur momentum.

## Auto-distillation

Une seconde famille de méthodes, l”**auto-distillation** (*self-distillation*), s’inspire de la distillation de connaissances (transfert des connaissances d’un grand modèle *enseignant* vers un plus petit modèle *élève*), mais sans modèle enseignant préexistant ni négatifs explicites.

**DINO** (*self-DIstillation with NO labels*, Caron et al., 2021) entraîne conjointement deux réseaux de même architecture, un **élève** et un **enseignant** :

- l’élève reçoit plusieurs vues augmentées d’une image et produit une distribution de probabilité (par un softmax) sur un ensemble de « pseudo-classes » ;  
- l’enseignant reçoit d’autres vues de la même image et produit de même une distribution cible ;  
- l’élève est entraîné, par descente de gradient, à faire correspondre sa distribution à celle de l’enseignant (entropie croisée) ;  
- l’enseignant n’est **jamais entraîné directement** : ses poids sont mis à jour comme une moyenne mobile exponentielle des poids de l’élève, exactement comme l’encodeur momentum de MoCo.  

## Question

Une paire (vue de l’élève, vue de l’enseignant) issue de la même image joue, dans DINO, un rôle analogue à quel élément de l’apprentissage contrastif ? DINO utilise-t-il des exemples négatifs ?

Les modèles **DINOv2** (Oquab et al., 2024), entraînés sur des centaines de millions d’images, produisent des représentations d’une qualité remarquable, utilisables directement (sans aucun réglage fin) pour de nombreuses tâches de vision, y compris la segmentation sémantique par simple analyse en composantes principales des caractéristiques.

## Modélisation masquée

La troisième grande famille de méthodes s’inspire directement du succès du pré-entraînement par masquage de mots dans le traitement du langage (BERT). Les **Masked Autoencoders** (MAE, He et al., 2022) adaptent ce principe aux images à l’aide d’une architecture Transformer (voir le [chapitre précédent](../../Bloc_deep2/session13/course.ipynb#chap-coursdeep6)) :

1. L’image est découpée en patchs, comme pour un Vision Transformer.  
1. Une **proportion très importante** des patchs (typiquement 75%) est masquée aléatoirement.  
1. Un encodeur Transformer ne traite **que les patchs visibles** (25% de la séquence), ce qui permet d’utiliser un encodeur de grande taille à coût de calcul maîtrisé.  
1. Un petit décodeur Transformer reçoit à la fois les représentations encodées des patchs visibles et des « jetons de masque » (*mask tokens*, un vecteur appris partagé) placés aux positions manquantes, puis reconstruit les pixels des patchs masqués.  
1. La fonction de coût (erreur quadratique moyenne) n’est calculée que sur les patchs masqués :  


$$
\mathcal L_{\text{MAE}} = \frac{1}{|\mathcal M|} \sum_{i \in \mathcal M} \lVert x_i - \hat x_i \rVert^2~,
$$

où $ \mathcal M $ est l’ensemble des indices des patchs masqués.

## Question

Pourquoi un taux de masquage aussi élevé (75%) est-il nécessaire pour que la tâche de reconstruction soit à la fois exigeante et pertinente ? On pourra comparer avec le taux de masquage typiquement utilisé pour le masquage de mots en traitement du langage (autour de 15%).

Après le pré-entraînement, seul l’encodeur est conservé (le décodeur, plus léger, est jeté), et ré-utilisé pour les tâches *downstream*, comme pour les méthodes contrastives.

## Évaluation des représentations auto-supervisées

Quelle que soit la méthode de pré-entraînement utilisée, la qualité des représentations apprises se mesure toujours par leur capacité à **transférer** à une tâche *downstream* étiquetée. Les trois protocoles d’évaluation les plus courants sont :

- le ***linear probing*** : l’encodeur pré-entraîné est gelé (ses poids ne sont plus mis à jour), et seule une couche linéaire est entraînée par-dessus pour la tâche *downstream* ;  
- le ***fine-tuning*** : l’ensemble du réseau (encodeur pré-entraîné et tête de classification) est ré-entraîné sur la tâche *downstream*, en partant des poids pré-entraînés plutôt que d’une initialisation aléatoire ;  
- l’évaluation en ***k plus proches voisins*** ($ k $-NN) directement dans l’espace des représentations gelées, sans aucun entraînement supplémentaire.  


Un résultat marquant, commun à toutes les méthodes de cette séance, est que les représentations auto-supervisées permettent d’obtenir de bonnes performances *downstream* avec **beaucoup moins d’exemples étiquetés** qu’un entraînement purement supervisé — c’est précisément ce que nous allons mesurer dans le TP de cette séance.